In [ ]:
import ast
import matplotlib.pyplot as plt
import pandas as pd

from dap_job_quality import PROJECT_DIR

In [ ]:
labelled_data = pd.read_csv(PROJECT_DIR / "inputs/labelled/Mapping evaluation - v2.csv", index_col=0)
lookup = pd.read_csv(PROJECT_DIR / "inputs/keyword_lookup - v4.csv")

In [ ]:
labelled_data.head()

In [ ]:
lookup['target_phrase'][0:10]

In [ ]:
lookup.head()

In [ ]:
lookup['subcategory'].value_counts(normalize=True)

In [ ]:
labelled_data['subcategory'].value_counts(normalize=True)

In [ ]:
# Calculate the average length in characters
lookup['char_count'] = lookup['target_phrase'].apply(len)
average_char_length = lookup['char_count'].mean()
print(f"Average length in characters: {average_char_length}")

# Calculate the average length in words
lookup['word_count'] = lookup['target_phrase'].apply(lambda x: len(x.split()))
average_word_length = lookup['word_count'].mean()
print(f"Average length in words: {average_word_length}")

In [ ]:

# Ensure the 'Correct?' column is correctly labeled and fix any issues
labelled_data['Correct?'] = labelled_data['Correct?'].replace({0: 'Incorrect', 1: 'Correct', '?': 'Unknown'})


# Calculate the count of correct and incorrect instances
count_correctness = labelled_data.groupby(['subcategory', 'Correct?']).size().reset_index(name='count')

# Pivot the table to have 'subcategory' as index and 'Correct?' as columns
count_correctness_pivot = count_correctness.pivot(index='subcategory', columns='Correct?', values='count').fillna(0)

# Plotting the count of correct and incorrect instances by subcategory
count_correctness_pivot.plot(kind='barh', stacked=True, figsize=(12, 8))
plt.title('Count of Correct and Incorrect Instances by Subcategory')
# plt.xlabel('Subcategory')
plt.xlabel('Count')
# plt.xticks(rotation=45)
plt.legend(title='Correctness')
plt.tight_layout()

# Save the plot as a file with bbox_inches='tight'
plt.savefig('count_correctness_by_subcategory.png', bbox_inches='tight')
plt.show()

In [ ]:
# Calculate the mean similarity score for each subcategory and correctness
mean_similarity = labelled_data.groupby(['subcategory', 'Correct?'])['similarity'].mean().reset_index()

mean_similarity

In [ ]:
correct_mean_cosine = labelled_data[labelled_data['Correct?']=='1']['similarity'].mean()
correct_mean_cosine

In [ ]:
pivot_df = mean_similarity.pivot(index='subcategory', columns='Correct?', values='similarity').fillna(0)

# Plotting the bar plot
pivot_df.plot(kind='barh', figsize=(12, 8), stacked=False)
plt.axvline(x=correct_mean_cosine, color='red', linestyle='--')
plt.title('Mean Similarity Score by Subcategory and Correctness')
plt.xlabel('Subcategory')
plt.ylabel('Mean Similarity Score')
plt.xticks(rotation=45)
plt.legend(title='Correctness')
plt.tight_layout()

In [ ]:
# Calculate proportional representation of each level of 'subcategory' in each dataframe
proportions_data = labelled_data['subcategory'].value_counts(normalize=True).sort_index()
proportions_lookup = lookup['subcategory'].value_counts(normalize=True).sort_index()

# Combine the proportions into a single dataframe for plotting
proportions = pd.DataFrame({'labelled data': proportions_data, 'lookup': proportions_lookup}).fillna(0)

# Plotting the comparison
proportions.plot(kind='barh', figsize=(10, 6))
# plt.title('Proportional Representation of Subcategories')
plt.xlabel('Proportion')
# plt.ylabel('Proportion')
plt.xticks(rotation=45)
plt.legend(title='Dataframe')
plt.savefig('proportional_representation.png', bbox_inches='tight')
plt.show()